In [2]:
from collections import defaultdict
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests


# Consente di eseguire il notebook sia dalla sua cartella sia dalla root del repository.
EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "eval.py").exists():
    EXPERIMENT_DIR = Path("3_experiments/experiment1").resolve()
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from eval import (
    ENCODER_SPLIT_PROTOCOL,
    INPUT_TYPES,
    MODEL_LABELS,
    OUTPUT_DIR,
    parse_prediction_info,
    readable_path,
    valid_evaluation_ids,
)

N_BOOTSTRAP = 10_000
N_PERMUTATIONS = 10_000
ALPHA = 0.05
BOOTSTRAP_SEED = 42
PERMUTATION_SEED = 43
BOOTSTRAP_OUTPUT = OUTPUT_DIR / "bootstrap_f1_results.csv"


def discover_common_evaluations():
    """Carica gli output attivi e calcola gli ID comuni nei quattro blocchi."""
    evaluations = []
    block_members = defaultdict(list)
    skipped = []

    prediction_roots = [OUTPUT_DIR / "decoders", OUTPUT_DIR / "encoders"]
    prediction_files = [
        file_path
        for root in prediction_roots
        if root.exists()
        for file_path in root.rglob("*.csv")
    ]

    for file_path in sorted(prediction_files):
        # INPUT_TYPES ? condiviso con eval.py; context_reply ? quindi escluso.
        if not any(input_name in file_path.stem for input_name in INPUT_TYPES):
            continue

        try:
            frame = pd.read_csv(readable_path(file_path))
            if "model_answer" not in frame.columns:
                continue

            relative_parts = file_path.relative_to(OUTPUT_DIR).parts
            if (
                relative_parts[0] == "encoders"
                and ENCODER_SPLIT_PROTOCOL not in relative_parts
            ):
                continue

            info = parse_prediction_info(file_path)
            if info["model"] not in MODEL_LABELS:
                continue

            if info["cohort"] == "Human held-out":
                reference = "Human majority vote"
                reference_column = "human_majority_label"
            else:
                reference = "Dataset ground truth"
                reference_column = "label"

            valid_ids = valid_evaluation_ids(frame, reference_column)
            if not valid_ids:
                raise ValueError("no rows with both prediction and reference")

            block_key = (
                info["cohort"],
                reference,
                info["dataset_label"],
            )
            evaluation = {
                "frame": frame,
                "info": info,
                "reference": reference,
                "reference_column": reference_column,
                "file_path": file_path,
                "block_key": block_key,
            }
            evaluations.append(evaluation)
            block_members[block_key].append(valid_ids)
        except (OSError, ValueError, pd.errors.ParserError) as error:
            skipped.append((file_path, str(error)))

    common_ids_by_block = {
        block_key: set.intersection(*valid_id_sets)
        for block_key, valid_id_sets in block_members.items()
    }
    empty_blocks = [key for key, ids in common_ids_by_block.items() if not ids]
    if empty_blocks:
        raise ValueError(f"No common valid predictions for blocks: {empty_blocks}")

    return evaluations, common_ids_by_block, skipped


def align_on_common_ids(evaluation, common_ids):
    """Allinea predizioni e riferimento per unique_id sul campione comune."""
    frame = evaluation["frame"]
    reference_column = evaluation["reference_column"]
    aligned = pd.DataFrame(
        {
            "unique_id": frame["unique_id"].astype("string"),
            "reference": pd.to_numeric(
                frame[reference_column], errors="coerce"
            ),
            "prediction": pd.to_numeric(
                frame["model_answer"], errors="coerce"
            ),
        }
    )
    aligned = aligned[aligned["unique_id"].isin(common_ids)].copy()
    if aligned["unique_id"].duplicated().any():
        raise ValueError(
            f"Duplicate unique_id in {evaluation['file_path']}"
        )

    aligned = aligned.set_index("unique_id").sort_index()
    expected_ids = sorted(common_ids)
    aligned = aligned.loc[expected_ids]
    aligned["reference"] = aligned["reference"].astype(int)
    aligned["prediction"] = aligned["prediction"].astype(int)
    return aligned


def positive_class_f1(references, predictions):
    true_positive = np.count_nonzero(
        (references == 1) & (predictions == 1)
    )
    false_positive = np.count_nonzero(
        (references == 0) & (predictions == 1)
    )
    false_negative = np.count_nonzero(
        (references == 1) & (predictions == 0)
    )
    denominator = (2 * true_positive) + false_positive + false_negative
    return 0.0 if denominator == 0 else (2 * true_positive) / denominator


def paired_resampling_f1_test(
    last_message,
    conversation,
    n_bootstrap=N_BOOTSTRAP,
    n_permutations=N_PERMUTATIONS,
    alpha=ALPHA,
    bootstrap_seed=BOOTSTRAP_SEED,
    permutation_seed=PERMUTATION_SEED,
):
    """Bootstrap CI and paired randomization test for the F1 difference."""
    if not last_message.index.equals(conversation.index):
        raise ValueError("The two inputs do not contain the same unique_id values")
    if not np.array_equal(
        last_message["reference"].to_numpy(),
        conversation["reference"].to_numpy(),
    ):
        raise ValueError("Reference labels differ between the paired inputs")

    references = last_message["reference"].to_numpy()
    last_predictions = last_message["prediction"].to_numpy()
    conversation_predictions = conversation["prediction"].to_numpy()
    n_samples = len(references)

    last_f1 = positive_class_f1(references, last_predictions)
    conversation_f1 = positive_class_f1(
        references, conversation_predictions
    )
    observed_delta = last_f1 - conversation_f1

    # Paired nonparametric bootstrap: resample unique_id values jointly.
    bootstrap_rng = np.random.default_rng(bootstrap_seed)
    bootstrap_deltas = np.empty(n_bootstrap, dtype=float)
    for iteration in range(n_bootstrap):
        sampled_indices = bootstrap_rng.integers(
            0, n_samples, size=n_samples
        )
        sampled_references = references[sampled_indices]
        bootstrap_deltas[iteration] = (
            positive_class_f1(
                sampled_references,
                last_predictions[sampled_indices],
            )
            - positive_class_f1(
                sampled_references,
                conversation_predictions[sampled_indices],
            )
        )

    ci_low, ci_high = np.quantile(
        bootstrap_deltas,
        [alpha / 2, 1 - (alpha / 2)],
    )

    # Paired randomization test under H0: within each unique_id, assignment
    # to last_u1_message/conversation is exchangeable.
    permutation_rng = np.random.default_rng(permutation_seed)
    null_deltas = np.empty(n_permutations, dtype=float)
    for iteration in range(n_permutations):
        swap = permutation_rng.integers(
            0, 2, size=n_samples, dtype=np.int8
        ).astype(bool)
        permuted_last = np.where(
            swap, conversation_predictions, last_predictions
        )
        permuted_conversation = np.where(
            swap, last_predictions, conversation_predictions
        )
        null_deltas[iteration] = (
            positive_class_f1(references, permuted_last)
            - positive_class_f1(references, permuted_conversation)
        )

    permutation_p_value = (
        np.count_nonzero(np.abs(null_deltas) >= abs(observed_delta)) + 1
    ) / (n_permutations + 1)

    return {
        "n_samples": n_samples,
        "f1_last_u1_message": last_f1,
        "f1_conversation": conversation_f1,
        "f1_delta": observed_delta,
        "bootstrap_delta_mean": float(bootstrap_deltas.mean()),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "ci_excludes_zero": bool(ci_low > 0 or ci_high < 0),
        "permutation_p_value": float(permutation_p_value),
        "significant_uncorrected": bool(permutation_p_value < alpha),
        "n_bootstrap": n_bootstrap,
        "n_permutations": n_permutations,
        "bootstrap_seed": bootstrap_seed,
        "permutation_seed": permutation_seed,
    }


def add_holm_corrections(results_frame, alpha=ALPHA):
    """Add Holm-adjusted p-values within blocks and across all tests."""
    corrected = results_frame.copy()
    corrected["p_value_holm_block"] = np.nan
    corrected["significant_holm_block"] = False
    corrected["holm_block_family_size"] = 0

    block_columns = ["cohort", "reference", "dataset"]
    for indices in corrected.groupby(block_columns).groups.values():
        indices = list(indices)
        rejected, adjusted, _, _ = multipletests(
            corrected.loc[indices, "permutation_p_value"].to_numpy(),
            alpha=alpha,
            method="holm",
        )
        corrected.loc[indices, "p_value_holm_block"] = adjusted
        corrected.loc[indices, "significant_holm_block"] = rejected
        corrected.loc[indices, "holm_block_family_size"] = len(indices)

    rejected, adjusted, _, _ = multipletests(
        corrected["permutation_p_value"].to_numpy(),
        alpha=alpha,
        method="holm",
    )
    corrected["p_value_holm_global"] = adjusted
    corrected["significant_holm_global"] = rejected
    corrected["holm_global_family_size"] = len(corrected)
    return corrected


def main():
    evaluations, common_ids_by_block, skipped = discover_common_evaluations()

    if skipped:
        print("Skipped prediction files:")
        for file_path, reason in skipped:
            print(f"  - {file_path}: {reason}")

    print("Common valid samples per evaluation block:")
    for block_key, common_ids in sorted(common_ids_by_block.items()):
        cohort, reference, dataset = block_key
        print(
            f"  - {cohort} | {reference} | {dataset}: "
            f"{len(common_ids)} samples"
        )

    pairs = defaultdict(dict)
    evaluation_lookup = {}
    for evaluation in evaluations:
        info = evaluation["info"]
        pair_key = (
            evaluation["block_key"],
            info["model"],
            info["technique"],
        )
        pairs[pair_key][info["input_type"]] = evaluation
        evaluation_lookup[pair_key] = info

    results = []
    for pair_key, input_evaluations in sorted(pairs.items()):
        if not {"last_u1_message", "conversation"}.issubset(
            input_evaluations
        ):
            continue

        block_key, model, technique = pair_key
        common_ids = common_ids_by_block[block_key]
        last_message = align_on_common_ids(
            input_evaluations["last_u1_message"],
            common_ids,
        )
        conversation = align_on_common_ids(
            input_evaluations["conversation"],
            common_ids,
        )
        test = paired_resampling_f1_test(last_message, conversation)
        info = evaluation_lookup[pair_key]
        cohort, reference, dataset = block_key
        results.append(
            {
                "cohort": cohort,
                "reference": reference,
                "dataset": dataset,
                "model": model,
                "model_label": info["model_label"],
                "technique": technique,
                **test,
            }
        )

    results_frame = pd.DataFrame(results).sort_values(
        ["cohort", "reference", "dataset", "model_label", "technique"]
    ).reset_index(drop=True)
    results_frame = add_holm_corrections(results_frame)
    results_frame.to_csv(BOOTSTRAP_OUTPUT, index=False)

    display_columns = [
        "cohort",
        "reference",
        "dataset",
        "model_label",
        "technique",
        "n_samples",
        "f1_last_u1_message",
        "f1_conversation",
        "f1_delta",
        "ci_low",
        "ci_high",
        "permutation_p_value",
        "p_value_holm_block",
        "significant_holm_block",
        "p_value_holm_global",
        "significant_holm_global",
    ]
    print(
        "\nPaired bootstrap CI + paired permutation test: "
        "last_u1_message - conversation"
    )
    print(
        results_frame[display_columns].to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}",
        )
    )
    print(f"\nResults saved to: {BOOTSTRAP_OUTPUT}")
    return results_frame


bootstrap_results = main()


Common valid samples per evaluation block:
  - Human held-out | Human majority vote | CANTRIP CSC: 92 samples
  - Human held-out | Human majority vote | CANTRIP MP: 85 samples
  - Standard test | Dataset ground truth | CANTRIP CSC: 422 samples
  - Standard test | Dataset ground truth | CANTRIP MP: 192 samples

Paired bootstrap CI + paired permutation test: last_u1_message - conversation
        cohort            reference     dataset       model_label        technique  n_samples  f1_last_u1_message  f1_conversation  f1_delta  ci_low  ci_high  permutation_p_value  p_value_holm_block  significant_holm_block  p_value_holm_global  significant_holm_global
Human held-out  Human majority vote CANTRIP CSC     EuroBERT-210M      Fine-Tuning         92              0.5289           0.4737    0.0552 -0.0758   0.1942               0.4487              1.0000                   False               1.0000                    False
Human held-out  Human majority vote CANTRIP CSC       Gemma-4-E4B Chain-